# Lipschitz Bounds for Convolutional Layers

This notebook is meant for developing the code which will enforce Lipschitz bounds on convolutional layers.

In [1]:
import sys
import os
import time
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import numpy as np

from itertools import product

# Add the project root to the path so we can import our modules
project_root = os.path.abspath('')
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Python path includes project root: {project_root in sys.path}")

Project root: /orcd/data/jhm/001/om2/rphess/projects/github.com/model_metamers_pytorch
Python path includes project root: True


In [2]:
# Code to compute the Lipschitz constant of a convolutional layer

# Code modified from https://github.com/blaisedelattre/lip4conv

def compute_sedghi_2019(X, n=None, return_time=True):
    """Estimate spectral norm of convolutional layer with Sedghi2019.

    From a convolutional filter, this function estimates the spectral norm of
    the convolutional layer for circular padding using Sedghi2019 [1]_.

    Code adapted from [2]_.

    This method should take long, but should be exact for circular padding filters.

    Parameters
    ----------
    X : ndarray, shape (cout, cint, h, w)
        Convolutional filter.
    n : None | int, default=None
        Size of input image. If None, n is set equal to k.
    return_time : bool, default True
        Return computational time.

    Returns
    -------
    sigma : float
        Largest singular value.
    time : float
        If `return_time` is True, it returns the computational time.

    References
    ----------
    .. [1] `The Singular Values of Convolutional Layers
        <https://arxiv.org/abs/1805.10408>`_
        H Sedghi, V Gupta & P M Long, ICLR, 2019
    .. [2] https://github.com/brain-research/conv-sv/blob/master/conv2d_singular_values.py
    """
    cout, cin, k, _ = X.shape
    if n is None:
        n = k
    start_time = time.time()
    X = torch.permute(X, (2, 3, 0, 1))
    fft_X = torch.fft.fft2(X, s=(n, n), dim=(0, 1))
    sigma = torch.linalg.matrix_norm(fft_X, ord=2, dim=(2, 3)).max()
    total_time = time.time() - start_time

    if return_time:
        return sigma, total_time
    else:
        return sigma


def normalize(arr):
    norm = torch.sqrt((arr**2).sum())
    return arr / (norm + 1e-12)


def compute_ryu_2019(X, n, n_iter=100, return_time=True):
    """Estimate spectral norm of convolutional layer with Ryu2019.

    From a convolutional filter, this function estimates the spectral norm of
    the convolutional layer for zero padding using Ryu2019 [1]_ or Farnia2019.

    Code adapted from [2]_.

    This method may take long, but should be a very good approximation for zero padding filters.

    Parameters
    ----------
    X : ndarray, shape (cout, cint, h, w)
        Convolutional filter.
    n : None | int
        Size of input image. If None, n is set equal to k.
    n_iter : int, default=100
        Number of iterations.
    return_time : bool, default True
        Return computational time.

    Returns
    -------
    sigma : float
        Largest singular value.
    time : float
        If `return_time` is True, it returns the computational time.

    References
    ----------
    .. [1] `Plug-and-Play Methods Provably Converge with Properly Trained
        Denoisers
        <https://par.nsf.gov/servlets/purl/10315555>`_
        E K Ryu, J Liu, S Wang, X Chen, Z Wang & W Yin, ICML, 2019
    .. [2] https://github.com/uclaopt/Provable_Plug_and_Play/blob/master/model/Spectral_Normalize.py

    """
    start_time = time.time()
    cout, cin, k, _ = X.shape
    if n is None:
        n = k
    input_size = (1, cin, n, n)
    u = torch.randn(input_size, dtype=X.dtype, device=X.device)
    u = u / u.norm(p=2)
    pad = (1, 1, 1, 1)
    pad_ = (-1, -1, -1, -1)
    for _ in range(n_iter):
        v = normalize(F.conv2d(F.pad(u, pad), X))
        u.data = normalize(F.pad(F.conv_transpose2d(v, X), pad_))
    u_hat, v_hat = u, v
    z = F.conv2d(F.pad(u_hat, pad), X)
    sigma = torch.mul(z, v_hat).sum()
    total_time = time.time() - start_time

    if return_time:
        return sigma, total_time
    else:
        return sigma


def compute_spectral_rescaling_conv(kernel, n_iter=1):
    if n_iter < 1:
        raise ValueError(f"n_iter must be at least equal to 1, got {n_iter}")
    effective_iter = 0
    kkt = kernel
    log_curr_norm = 0
    for _ in range(n_iter):
        padding = kkt.shape[-1] - 1
        kkt_norm = kkt.norm().detach()
        kkt = kkt / kkt_norm
        log_curr_norm = 2 * (log_curr_norm + kkt_norm.log())
        kkt = F.conv2d(kkt, kkt, padding=padding)
        effective_iter += 1
    inverse_power = 2**(-effective_iter)
    t = torch.abs(kkt)
    t = t.sum(dim=(1, 2, 3)).pow(inverse_power)
    norm = torch.exp(log_curr_norm * inverse_power)
    t = t*norm
    return t


def compute_delattre2024(X, n_iter=4, return_time=True):
    """Estimate spectral norm of convolutional layer with Delattre2024.

    From a convolutional filter, this function estimates the spectral norm of
    the convolutional layer with zero padding using Delattre2024 [1]_.

    Parameters
    ----------
    X : ndarray, shape (cout, cint, k, k)
        Convolutional filter.
    n_iter : int, default=4
        Number of iterations.
    return_time : bool, default True
        Return computational time.

    Returns
    -------
    sigma : float
        Largest singular value.
    time : float
        If `return_time` is True, it returns the computational time.

    References
    ----------
    .. [1] `Spectral Norm of Convolutional Layers with Circular and Zero Paddings
        <https://arxiv.org/abs/2402.00240>`_
        B Delattre, Q Barthélemy & A Allauzen, arXiv, 2024
    """
    cout, cin, _, _ = X.shape
    if cin > cout:
        X = X.transpose(0, 1)
    start_time = time.time()
    rescale_weights = compute_spectral_rescaling_conv(X, n_iter)
    sigma = rescale_weights.max()
    total_time = time.time() - start_time

    if return_time:
        return sigma, total_time
    else:
        return sigma


def compute_delattre2023(X, n=None, n_iter=4, return_time=True):
    """Estimate spectral norm of convolutional layer with Delattre2023.

    From a convolutional filter, this function estimates the spectral norm of
    the convolutional layer for circular padding using [Section.3, Algo. 3] Delattre2023.

    Parameters
    ----------
    X : ndarray, shape (cout, cint, k, k)
        Convolutional filter.
    n : None | int, default=None
        Size of input image. If None, n is set equal to k.
    n_iter : int, default=4
        Number of iterations.
    return_time : bool, default True
        Return computational time.

    Returns
    -------
    sigma : float
        Largest singular value.
    time : float
        If `return_time` is True, it returns the computational time.
    """
    cout, cin, k, _ = X.shape
    if n is None:
        n = k
    if cin > cout:
        X = X.transpose(0, 1)
        cin, cout = cout, cin
    start_time = time.time()

    crossed_term = (
        torch.fft.rfft2(X, s=(n, n)).reshape(cout, cin, -1).permute(2, 0, 1)
    )
    inverse_power = 1
    log_curr_norm = torch.zeros(crossed_term.shape[0]).cuda()
    for _ in range(n_iter):
        norm_crossed_term = crossed_term.norm(dim=(1, 2))
        crossed_term /= norm_crossed_term.reshape(-1, 1, 1)
        log_curr_norm = 2 * log_curr_norm + norm_crossed_term.log()
        crossed_term = torch.bmm(crossed_term.conj().transpose(1, 2), crossed_term)
        inverse_power /= 2
    sigma = (
        crossed_term.norm(dim=(1, 2)).pow(inverse_power)
        * ((2 * inverse_power * log_curr_norm).exp())
    ).max()
    total_time = time.time() - start_time

    if return_time:
        return sigma, total_time
    else:
        return sigma


class GramIterationConvBackward(torch.autograd.Function):
    @staticmethod
    def forward(ctx, rfft_kernel, n_iter):
        with torch.no_grad():
            inverse_power = 1
            Gt = rfft_kernel
            log_curr_norm = 0.0
            for _ in range(n_iter):
                norm_Gt = Gt.norm(dim=(1, 2))
                Gt = Gt / norm_Gt.reshape(-1, 1, 1)
                log_curr_norm = 2 * (log_curr_norm + norm_Gt.log())
                Gt = torch.bmm(Gt.conj().transpose(1, 2), Gt)
                inverse_power /= 2

            sqrt_Gt_norm = Gt.norm(dim=(1, 2)).pow(inverse_power) * (
                (inverse_power * log_curr_norm).exp()
            )
            max_sqrt_Gt_norm, idx = sqrt_Gt_norm.max(dim=0)

            ctx.save_for_backward(
                rfft_kernel[idx, :, :].detach(), Gt[idx, :, :].detach()
            )
            ctx.idx = idx.item()
            ctx.norm_Gt = max_sqrt_Gt_norm.detach()
            ctx.n_iter = n_iter
            ctx.dtype = rfft_kernel.dtype
            ctx.device = rfft_kernel.device
            ctx.shape_input = rfft_kernel.shape

            return max_sqrt_Gt_norm

    @staticmethod
    def backward(ctx, grad_output):
        G, Gt = ctx.saved_tensors
        idx = ctx.idx
        n_iter = ctx.n_iter
        grad_input = torch.zeros(ctx.shape_input, dtype=ctx.dtype, device=ctx.device)
        if n_iter == 0:
            jac = G / ctx.norm_Gt
        else:
            norm_G_sq = ctx.norm_Gt**2
            Gdag_G = (G.conj().t() @ G) / norm_G_sq
            num = G @ Gdag_G.matrix_power(2**n_iter - 1)
            denom = Gdag_G.matrix_power(2 ** (n_iter - 1)).norm().pow(2 - 0.5**n_iter)
            jac = (num / denom) * norm_G_sq ** (-0.5)
        grad_input[idx, :, :] = jac
        return grad_output * grad_input, None


gram_iteration_conv_backward = GramIterationConvBackward.apply


def compute_delattre2023_backward(kernel, n, n_iter=4, return_time=False):
    """Estimate spectral norm of convolutional layer with Delattre2023.

    From a convolutional filter, this function estimates the spectral norm of
    the convolutional layer for circular padding using 
    [Section.3, Algo. 3] Delattre2023 [1]_, with explicit backward implementation.

    Parameters
    ----------
    X : ndarray, shape (cout, cint, k, k)
        Convolutional filter.
    n : None | int, default=None
        Size of input image. If None, n is set equal to k.
    n_iter : int, default=4
        Number of iterations.
    return_time : bool, default True
        Return computational time.

    Returns
    -------
    sigma : float
        Largest singular value.
    time : float
        If `return_time` is True, it returns the computational time.

    References
    ----------
    .. [1] `Efficient Bound of Lipschitz Constant for Convolutional Layers
        by Gram Iteration
        <https://arxiv.org/abs/2305.16173>`_
        B Delattre, Q Barthélemy, A Araujo & A Allauzen, ICML, 2023
    """
    cout, cin, k, _ = kernel.shape
    if n is None:
        n = k
    start_time = time.time()
    if cin > cout:
        kernel = kernel.transpose(0, 1)
        cin, cout = cout, cin
    crossed_term = torch.fft.rfft2(kernel, s=(n, n))
    crossed_term = crossed_term.reshape(cout, cin, -1).permute(2, 0, 1)
    sigma = gram_iteration_conv_backward(crossed_term, n_iter)
    total_time = time.time() - start_time

    if return_time:
        return sigma, total_time
    else:
        return sigma

In [3]:
def general_conv_to_matrix(conv_weight, input_shape, stride=1, padding=0):
    """
    Converts a general 4D convolution tensor into a structured sparse matrix
    representing the equivalent linear transformation on a flattened input image.

    Parameters:
    - conv_weight: np.ndarray of shape (out_channels, in_channels, kH, kW)
    - input_shape: tuple (in_channels, H, W) of the input image (single example)
    - stride: int or tuple (stride_H, stride_W)
    - padding: int or tuple (pad_H, pad_W)

    Returns:
    - M: numpy.ndarray of shape (out_channels * out_H * out_W, in_channels * H * W)
    """
    out_channels, in_channels, kH, kW = conv_weight.shape
    C, H, W = input_shape
    if isinstance(stride, int):
        stride_H, stride_W = stride, stride
    else:
        stride_H, stride_W = stride

    if isinstance(padding, int):
        pad_H, pad_W = padding, padding
    else:
        pad_H, pad_W = padding

    # Calculate output dimensions with padding
    out_H = (H + 2 * pad_H - kH) // stride_H + 1
    out_W = (W + 2 * pad_W - kW) // stride_W + 1

    # Initialize dense matrix with zeros
    M = np.zeros((out_channels * out_H * out_W, C * H * W))

    row = 0
    for oc in range(out_channels):
        for oh in range(out_H):
            for ow in range(out_W):
                h_start = oh * stride_H - pad_H  # Account for padding
                w_start = ow * stride_W - pad_W  # Account for padding
                for ic in range(in_channels):
                    for kh in range(kH):
                        for kw in range(kW):
                            ih = h_start + kh
                            iw = w_start + kw
                            # Check if we're within the actual input bounds (not padding)
                            if 0 <= ih < H and 0 <= iw < W:
                                input_idx = ic * H * W + ih * W + iw
                                weight_val = conv_weight[oc, ic, kh, kw]
                                M[row, input_idx] = weight_val
                row += 1

    M = torch.from_numpy(M)
    M = M.to(torch.float32)
    return M

In [4]:
# Let's debug with a very simple case first
print("=== DEBUGGING SIMPLE CASE ===")

# Simple 1-channel, 3x3 input, 2x2 kernel, no padding, stride 1
input_simple = torch.randn(1, 1, 3, 3)
kernel_simple = torch.randn(1, 1, 2, 2)

print("Input shape:", input_simple.shape)
print("Kernel shape:", kernel_simple.shape)
print("Input values:")
print(input_simple[0, 0])
print("Kernel values:")
print(kernel_simple[0, 0])

# Standard PyTorch convolution
output_pytorch = F.conv2d(input_simple, kernel_simple, padding=0, stride=1)
print("\nPyTorch output shape:", output_pytorch.shape)
print("PyTorch output:")
print(output_pytorch[0, 0])

# Our matrix method
flattened_input_simple = input_simple.view(1, 1*3*3)
flattened_kernel_simple = general_conv_to_matrix(kernel_simple, (1, 3, 3), stride=1, padding=0)

print("\nFlattened input shape:", flattened_input_simple.shape)
print("Flattened kernel shape:", flattened_kernel_simple.shape)
print("Flattened kernel:")
print(flattened_kernel_simple)

output_custom_simple = flattened_input_simple @ flattened_kernel_simple.T
output_custom_simple = output_custom_simple.view(1, 1, 2, 2)

print("\nCustom output shape:", output_custom_simple.shape)
print("Custom output:")
print(output_custom_simple[0, 0])

print("\nDo they match?", torch.allclose(output_pytorch, output_custom_simple, atol=1e-6))


=== DEBUGGING SIMPLE CASE ===
Input shape: torch.Size([1, 1, 3, 3])
Kernel shape: torch.Size([1, 1, 2, 2])
Input values:
tensor([[ 1.9591, -0.4231, -0.4155],
        [ 0.2683,  0.1564,  0.3306],
        [ 2.2484,  1.0758, -0.5805]])
Kernel values:
tensor([[-0.5178, -0.3416],
        [ 0.0228,  1.3164]])

PyTorch output shape: torch.Size([1, 1, 2, 2])
PyTorch output:
tensor([[-0.6580,  0.7997],
        [ 1.2750, -0.9336]])

Flattened input shape: torch.Size([1, 9])
Flattened kernel shape: torch.Size([4, 9])
Flattened kernel:
tensor([[-0.5178, -0.3416,  0.0000,  0.0228,  1.3164,  0.0000,  0.0000,  0.0000,
          0.0000],
        [ 0.0000, -0.5178, -0.3416,  0.0000,  0.0228,  1.3164,  0.0000,  0.0000,
          0.0000],
        [ 0.0000,  0.0000,  0.0000, -0.5178, -0.3416,  0.0000,  0.0228,  1.3164,
          0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.5178, -0.3416,  0.0000,  0.0228,
          1.3164]])

Custom output shape: torch.Size([1, 1, 2, 2])
Custom output:
tensor(

In [5]:
# Let's also try a manual calculation to verify
print("\n=== MANUAL VERIFICATION ===")

# For a 3x3 input with 2x2 kernel, let's manually compute one output element
# Input: [[a, b, c], [d, e, f], [g, h, i]]
# Kernel: [[k1, k2], [k3, k4]]
# Output[0,0] should be: a*k1 + b*k2 + d*k3 + e*k4

input_manual = torch.tensor([[[[1., 2., 3.], 
                               [4., 5., 6.], 
                               [7., 8., 9.]]]])
kernel_manual = torch.tensor([[[[0.1, 0.2], 
                                [0.3, 0.4]]]])

print("Manual input:")
print(input_manual[0, 0])
print("Manual kernel:")
print(kernel_manual[0, 0])

# PyTorch result
pytorch_result = F.conv2d(input_manual, kernel_manual, padding=0, stride=1)
print("\nPyTorch result:")
print(pytorch_result[0, 0])

# Expected result (manual calculation)
expected_00 = 1*0.1 + 2*0.2 + 4*0.3 + 5*0.4  # top-left output
expected_01 = 2*0.1 + 3*0.2 + 5*0.3 + 6*0.4  # top-right output
expected_10 = 4*0.1 + 5*0.2 + 7*0.3 + 8*0.4  # bottom-left output
expected_11 = 5*0.1 + 6*0.2 + 8*0.3 + 9*0.4  # bottom-right output

print(f"\nExpected values:")
print(f"[0,0]: {expected_00}")
print(f"[0,1]: {expected_01}")
print(f"[1,0]: {expected_10}")
print(f"[1,1]: {expected_11}")

# Our matrix method
flattened_input_manual = input_manual.view(1, 9)
flattened_kernel_manual = general_conv_to_matrix(kernel_manual, (1, 3, 3), stride=1, padding=0)

print(f"\nOur flattened input: {flattened_input_manual}")
print(f"Our matrix shape: {flattened_kernel_manual.shape}")
print(f"Our matrix:\n{flattened_kernel_manual}")

output_manual = flattened_input_manual @ flattened_kernel_manual.T
output_manual = output_manual.view(1, 1, 2, 2)

print(f"\nOur result:")
print(output_manual[0, 0])

print(f"\nDo manual calculations match PyTorch? {torch.allclose(pytorch_result, output_manual, atol=1e-6)}")



=== MANUAL VERIFICATION ===
Manual input:
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
Manual kernel:
tensor([[0.1000, 0.2000],
        [0.3000, 0.4000]])

PyTorch result:
tensor([[3.7000, 4.7000],
        [6.7000, 7.7000]])

Expected values:
[0,0]: 3.7
[0,1]: 4.7
[1,0]: 6.7
[1,1]: 7.699999999999999

Our flattened input: tensor([[1., 2., 3., 4., 5., 6., 7., 8., 9.]])
Our matrix shape: torch.Size([4, 9])
Our matrix:
tensor([[0.1000, 0.2000, 0.0000, 0.3000, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.1000, 0.2000, 0.0000, 0.3000, 0.4000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.1000, 0.2000, 0.0000, 0.3000, 0.4000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.1000, 0.2000, 0.0000, 0.3000, 0.4000]])

Our result:
tensor([[3.7000, 4.7000],
        [6.7000, 7.7000]])

Do manual calculations match PyTorch? True


In [6]:
# set up dummy input and kernel  
input = torch.randn(1, 2, 16, 16)
kernel = torch.randn(4, 2, 2, 2)

print("Input shape:", input.shape)
print("Kernel shape:", kernel.shape)

# compute the output of the convolutional layer using standard convolution
output_standard = F.conv2d(input, kernel, padding=0, stride=1)

# compute the output of the convolutional layer using our custom convolution flattening
flattened_input = input.view(1, 2*16*16)
flattened_kernel = general_conv_to_matrix(kernel, (2, 16, 16), stride=1, padding=0)  # Add padding=0
output_custom = flattened_input @ flattened_kernel.T
output_custom = output_custom.view(1, 4, 15, 15)

# check that the outputs are the same
print("Standard output shape:", output_standard.shape)
print("Custom output shape:", output_custom.shape)
print("Do they match?", torch.allclose(output_standard, output_custom))

# Let's check the max difference to see how close they are
if not torch.allclose(output_standard, output_custom):
    diff = torch.abs(output_standard - output_custom)
    print(f"Max difference: {diff.max().item()}")
    print(f"Mean difference: {diff.mean().item()}")
    print(f"Are they close with higher tolerance?", torch.allclose(output_standard, output_custom, atol=1e-4))

Input shape: torch.Size([1, 2, 16, 16])
Kernel shape: torch.Size([4, 2, 2, 2])
Standard output shape: torch.Size([1, 4, 15, 15])
Custom output shape: torch.Size([1, 4, 15, 15])
Do they match? False
Max difference: 9.5367431640625e-07
Mean difference: 1.0644102133028355e-07
Are they close with higher tolerance? True


In [7]:
# Test with stride=2 as originally requested
print("\n=== TESTING WITH STRIDE=2 ===")

input_stride2 = torch.randn(1, 2, 16, 16)
kernel_stride2 = torch.randn(4, 2, 3, 3)  # Using 3x3 kernel for stride=2

print("Input shape:", input_stride2.shape)
print("Kernel shape:", kernel_stride2.shape)

# compute the output of the convolutional layer using standard convolution with stride=2
output_standard_stride2 = F.conv2d(input_stride2, kernel_stride2, padding=1, stride=2)

# compute the output of the convolutional layer using our custom convolution flattening
flattened_input_stride2 = input_stride2.view(1, 2*16*16)
flattened_kernel_stride2 = general_conv_to_matrix(kernel_stride2, (2, 16, 16), stride=2, padding=1)
output_custom_stride2 = flattened_input_stride2 @ flattened_kernel_stride2.T

# Calculate expected output shape: (16 + 2*1 - 3) // 2 + 1 = 8
expected_out_size = (16 + 2*1 - 3) // 2 + 1
output_custom_stride2 = output_custom_stride2.view(1, 4, expected_out_size, expected_out_size)

# check that the outputs are the same
print("Standard output shape:", output_standard_stride2.shape)
print("Custom output shape:", output_custom_stride2.shape)
print("Do they match?", torch.allclose(output_standard_stride2, output_custom_stride2))

# Let's check the max difference to see how close they are
if not torch.allclose(output_standard_stride2, output_custom_stride2):
    diff = torch.abs(output_standard_stride2 - output_custom_stride2)
    print(f"Max difference: {diff.max().item()}")
    print(f"Mean difference: {diff.mean().item()}")
    print(f"Are they close with higher tolerance?", torch.allclose(output_standard_stride2, output_custom_stride2, atol=1e-4))



=== TESTING WITH STRIDE=2 ===
Input shape: torch.Size([1, 2, 16, 16])
Kernel shape: torch.Size([4, 2, 3, 3])
Standard output shape: torch.Size([1, 4, 8, 8])
Custom output shape: torch.Size([1, 4, 8, 8])
Do they match? True


In [15]:
kernel = torch.randn(2, 1, 2, 2)
flattened_kernel = general_conv_to_matrix(kernel, (1, 2, 2), stride=1, padding=1)

# compute the svd of the flattened kernel
U, S, V = torch.svd(flattened_kernel)
spec_norm = S.abs().max()

estimate = compute_delattre2024(kernel, n_iter=4, return_time=False)

print(f"Spectral norm: {spec_norm}")
print(f"Estimate: {estimate}")

Spectral norm: 2.365321159362793
Estimate: 2.991854667663574
